# Olfacto — EDA, baseline e inferência

Notebook público do projeto **Olfacto**: predição de odor molecular a partir de SMILES.

- Dataset: [SamuelCagnani/olfacto-odor-138-ptbr](https://huggingface.co/datasets/SamuelCagnani/olfacto-odor-138-ptbr)
- Modelo: [SamuelCagnani/olfacto-chemberta-138](https://huggingface.co/SamuelCagnani/olfacto-chemberta-138)
- Código: [github.com/cagnani/olfacto-ai](https://github.com/cagnani/olfacto-ai)

O objetivo é explorar o desbalanceamento dos 138 rótulos, treinar um baseline clássico
(Morgan + regressão logística) e demonstrar a inferência do ChemBERTa publicado.

Roda em CPU. Ative a internet no Kaggle.

In [ ]:
!pip install -q iterative-stratification

In [ ]:
import json

from pathlib import Path

import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.multiclass import OneVsRestClassifier
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

RDLogger.DisableLog("rdApp.*")

DATASET_ID = "SamuelCagnani/olfacto-odor-138-ptbr"
MODEL_ID = "SamuelCagnani/olfacto-chemberta-138"

path = hf_hub_download(
    DATASET_ID, "curated_GS_LF_merged_4983.csv", repo_type="dataset"
)
df = pd.read_csv(path)
labels = list(df.columns[2:])
print("formato:", df.shape)
print("rótulos:", len(labels))

## Desbalanceamento

O rótulo mais comum aparece em 1.902 moléculas; o mais raro, em 31. Um split aleatório
faria os rótulos raros sumirem da validação.

In [ ]:
counts = df[labels].sum().sort_values(ascending=False)

print("mais comuns:")
print(counts.head(10).to_string())
print("\nmais raros:")
print(counts.tail(10).to_string())
print("\nrótulos por molécula:")
print(df[labels].sum(axis=1).describe().to_string())

## Baseline: Morgan + regressão logística

Fingerprints de 2.048 bits (raio 2) com uma regressão logística balanceada por rótulo.
O split é multi-label estratificado, para preservar os rótulos raros.

In [ ]:
def morgan_matrix(smiles, n_bits=2048, radius=2):
    rows = []
    for smi in smiles:
        molecule = Chem.MolFromSmiles(smi)
        fingerprint = AllChem.GetMorganFingerprintAsBitVect(
            molecule, radius, nBits=n_bits
        )
        rows.append(np.array(list(fingerprint), dtype=np.uint8))
    return np.vstack(rows)


X = morgan_matrix(df["nonStereoSMILES"].tolist())
y = df[labels].to_numpy()

splitter = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(splitter.split(X, y))

classifier = OneVsRestClassifier(
    LogisticRegression(max_iter=500, class_weight="balanced"), n_jobs=-1
)
classifier.fit(X[train_idx], y[train_idx])
scores = classifier.predict_proba(X[test_idx])

print("treino:", len(train_idx), "| teste:", len(test_idx))
print("micro ROC-AUC:", round(roc_auc_score(y[test_idx], scores, average="micro"), 4))

In [ ]:
rows = []
for index, label in enumerate(labels):
    truth = y[test_idx, index]
    if truth.min() == truth.max():
        continue
    rows.append(
        {
            "label": label,
            "positives": int(y[:, index].sum()),
            "auc": roc_auc_score(truth, scores[:, index]),
        }
    )

per_label = pd.DataFrame(rows).sort_values("auc", ascending=False)
print("macro ROC-AUC:", round(per_label["auc"].mean(), 4))
per_label.head(10)

## Inferência do ChemBERTa

O modelo publicado foi treinado em todas as 4.983 moléculas, então esta seção é uma
demonstração de inferência, não uma avaliação em teste. Para o número out-of-fold, veja o
model card e o artigo.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
model.eval()

label_order = json.loads(
    Path(hf_hub_download(MODEL_ID, "labels.json")).read_text(encoding="utf-8")
)
lexicon = json.loads(
    Path(
        hf_hub_download(DATASET_ID, "odor_labels_ptbr.json", repo_type="dataset")
    ).read_text(encoding="utf-8")
)

examples = {
    "etanol": "CCO",
    "vanilina": "COC1=C(C=CC(=C1)C=O)O",
    "limoneno": "CC1=CCC(CC1)C(=C)C",
    "alicina": "C=CCSS(=O)CC=C",
}

for name, smiles in examples.items():
    inputs = tokenizer(
        smiles, return_tensors="pt", truncation=True, max_length=128
    )
    with torch.no_grad():
        probabilities = torch.sigmoid(model(**inputs).logits)[0]
    top = torch.topk(probabilities, 5)
    result = [
        (lexicon[label_order[int(index)]]["pt_br"], round(float(value), 2))
        for value, index in zip(top.values, top.indices)
    ]
    print(name, result)

## Conclusões

- O baseline Morgan + LogReg é rápido e forte: ~0,83 de ROC-AUC macro out-of-fold no
  projeto completo.
- O ChemBERTa sozinho fica em 0,7516 (OOF); o ensemble com fingerprint chega a 0,8402.
- O modelo foi treinado em catálogos de fragrância; ingredientes salgados podem ter
  previsões fracas.
- O projeto completo, com avaliação e agente, está em
  [github.com/cagnani/olfacto-ai](https://github.com/cagnani/olfacto-ai).